<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 11


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Customer в C#, который будет представлять информацию о
клиентах или покупателях. На основе этого класса разработать 2-3 производных
класса, демонстрирующих принципы наследования и полиморфизма. В каждом из
классов должны быть реализованы новые атрибуты и методы, а также
переопределены некоторые методы базового класса для демонстрации
полиморфизма.

Требования к базовому классу Customer:

• Атрибуты: Идентификатор клиента (CustomerId), Имя (Name), Электронная
почта (Email).

• Методы:

1. GetFullName(): метод для получения полного имени клиента.
2.  UpdateEmail(string newEmail): метод для обновления электронной
почты клиента.
3.  ViewProfile(): метод для просмотра профиля клиента.

Требования к производным классам:
1. VIPКлиент (VipCustomer): Должен содержать дополнительные атрибуты,
такие как Баланс лояльности (LoyaltyPoints). Метод ViewProfile() должен быть
переопределен для отображения дополнительной информации о VIPклиенте.
2. ОбычныйКлиент (RegularCustomer): Должен содержать дополнительные
атрибуты, такие как Дата регистрации (RegistrationDate).
Метод UpdateEmail() должен быть переопределен для добавления
информации о дате последнего обновления электронной почты.
3. ГрупповойКлиент (GroupCustomer) (если требуется третий класс): Должен
содержать дополнительные атрибуты, такие как Название группы
(GroupName). Метод GetFullName() должен быть переопределен для
отображения названия группы вместо имени клиента.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [2]:
using System;
using System.Collections.Generic;
using System.Linq;

public interface INotifiable
{
    void SendNotification(string message);
    string PhoneNumber { get; set; }
    bool IsSubscribed { get; set; }
}

public interface IAuthenticable
{
    bool Authenticate(string password);
    void ChangePassword(string newPassword);
    DateTime LastLogin { get; set; }
}

public interface IPurchasable
{
    void MakePurchase(double amount);
    double GetTotalSpent();
    int GetPurchaseCount();
}

public class CustomerManager<T> where T : Customer
{
    private List<T> _customers = new List<T>();
    private readonly INotificationService _notificationService;
    private readonly IAuthenticationService _authService;
    
    public CustomerManager(INotificationService notificationService, IAuthenticationService authService)
    {
        _notificationService = notificationService;
        _authService = authService;
    }
    
    public void AddCustomer(T customer)
    {
        _customers.Add(customer);
        Console.WriteLine($"Добавлен клиент: {customer.GetName()}");
    }
    
    public bool RemoveCustomer(int customerId)
    {
        var customer = _customers.Find(c => c.GetCustomerId() == customerId);
        if (customer != null)
        {
            _customers.Remove(customer);
            Console.WriteLine($"Удален клиент: {customer.GetName()}");
            return true;
        }
        return false;
    }
    
    public T FindCustomerById(int customerId)
    {
        return _customers.Find(c => c.GetCustomerId() == customerId);
    }
    
    public T FindCustomerByName(string name)
    {
        return _customers.Find(c => c.GetName().Equals(name, StringComparison.OrdinalIgnoreCase));
    }
    
    public void DisplayAllCustomers()
    {
        Console.WriteLine($"\n=== Все клиенты ({typeof(T).Name}) ===");
        foreach (var customer in _customers)
        {
            customer.DisplayProfile();
            Console.WriteLine("---");
        }
    }
    
    public int CustomerCount => _customers.Count;
    
    public void SendBulkNotification(string message)
    {
        Console.WriteLine($"\nМассовая рассылка для {_customers.Count} клиентов:");
        foreach (var customer in _customers)
        {
            if (customer is INotifiable notifiable && notifiable.IsSubscribed)
            {
                _notificationService.SendNotification(customer.GetName(), message, notifiable.PhoneNumber);
            }
        }
    }
    
    public bool AuthenticateCustomer(T customer, string password)
    {
        return _authService.Authenticate(customer.GetName(), password);
    }
}

public class LoyaltySystem<T> where T : Customer
{
    private Dictionary<int, int> _points = new Dictionary<int, int>();
    private readonly IPurchaseService _purchaseService;
    
    public LoyaltySystem(IPurchaseService purchaseService)
    {
        _purchaseService = purchaseService;
    }
    
    public void AddPoints(T customer, int points)
    {
        int customerId = customer.GetCustomerId();
        if (_points.ContainsKey(customerId))
        {
            _points[customerId] += points;
        }
        else
        {
            _points[customerId] = points;
        }
        Console.WriteLine($"Начислено {points} баллов клиенту {customer.GetName()}");
    }
    
    public int GetPoints(T customer)
    {
        return _points.ContainsKey(customer.GetCustomerId()) ? _points[customer.GetCustomerId()] : 0;
    }
    
    public bool RedeemPoints(T customer, int points)
    {
        int customerId = customer.GetCustomerId();
        if (_points.ContainsKey(customerId) && _points[customerId] >= points)
        {
            _points[customerId] -= points;
            Console.WriteLine($"Списано {points} баллов у клиента {customer.GetName()}");
            return true;
        }
        Console.WriteLine($"Недостаточно баллов у клиента {customer.GetName()}");
        return false;
    }
    
    public void ProcessPurchase(T customer, double amount)
    {
        _purchaseService.ProcessPurchase(customer.GetName(), amount);
        int pointsEarned = (int)(amount / 10);
        AddPoints(customer, pointsEarned);
    }
}

public interface INotificationService
{
    void SendNotification(string customerName, string message, string phoneNumber);
}

public interface IAuthenticationService
{
    bool Authenticate(string username, string password);
}

public interface IPurchaseService
{
    void ProcessPurchase(string customerName, double amount);
}

public class EmailNotificationService : INotificationService
{
    public void SendNotification(string customerName, string message, string phoneNumber)
    {
        Console.WriteLine($"Email отправлен {customerName}: {message}");
        if (!string.IsNullOrEmpty(phoneNumber))
        {
            Console.WriteLine($"SMS отправлено на {phoneNumber}: {message}");
        }
    }
}

public class BasicAuthenticationService : IAuthenticationService
{
    public bool Authenticate(string username, string password)
    {
        bool isAuthenticated = password.Length >= 6;
        Console.WriteLine($"Аутентификация {username}: {(isAuthenticated ? "УСПЕХ" : "ОШИБКА")}");
        return isAuthenticated;
    }
}

public class PurchaseProcessingService : IPurchaseService
{
    public void ProcessPurchase(string customerName, double amount)
    {
        Console.WriteLine($"Обработка покупки для {customerName} на сумму {amount:N2}");
    }
}

public delegate void CustomerEventHandler(object sender, CustomerEventArgs e);
public delegate void PurchaseEventHandler(object sender, PurchaseEventArgs e);
public delegate void NotificationEventHandler(string message, Customer customer);

public class CustomerEventArgs : EventArgs
{
    public string Message { get; }
    public DateTime EventTime { get; }
    
    public CustomerEventArgs(string message)
    {
        Message = message;
        EventTime = DateTime.Now;
    }
}

public class PurchaseEventArgs : EventArgs
{
    public double Amount { get; }
    public string Product { get; }
    public DateTime PurchaseTime { get; }
    
    public PurchaseEventArgs(double amount, string product)
    {
        Amount = amount;
        Product = product;
        PurchaseTime = DateTime.Now;
    }
}

public class CustomerEventService
{
    public event CustomerEventHandler CustomerRegistered;
    public event CustomerEventHandler CustomerUpdated;
    public event PurchaseEventHandler PurchaseMade;
    public event NotificationEventHandler BulkNotificationSent;
    
    public void OnCustomerRegistered(Customer customer)
    {
        CustomerRegistered?.Invoke(this, new CustomerEventArgs($"Зарегистрирован новый клиент: {customer.GetName()}"));
    }
    
    public void OnCustomerUpdated(Customer customer)
    {
        CustomerUpdated?.Invoke(this, new CustomerEventArgs($"Обновлены данные клиента: {customer.GetName()}"));
    }
    
    public void OnPurchaseMade(Customer customer, double amount, string product)
    {
        PurchaseMade?.Invoke(this, new PurchaseEventArgs(amount, product));
    }
    
    public void OnBulkNotificationSent(string message, Customer customer)
    {
        BulkNotificationSent?.Invoke(message, customer);
    }
}

public class CustomerRepository
{
    private Dictionary<int, Customer> _customersById;
    private Dictionary<string, List<Customer>> _customersByCity;
    private HashSet<string> _customerEmails;
    private List<Customer> _recentlyActive;
    private Queue<Customer> _registrationQueue;
    private Stack<Customer> _lastViewed;
    
    public CustomerRepository()
    {
        _customersById = new Dictionary<int, Customer>();
        _customersByCity = new Dictionary<string, List<Customer>>();
        _customerEmails = new HashSet<string>();
        _recentlyActive = new List<Customer>();
        _registrationQueue = new Queue<Customer>();
        _lastViewed = new Stack<Customer>();
    }
    
    public void AddCustomer(Customer customer)
    {
        _customersById[customer.GetCustomerId()] = customer;
        
        string city = ExtractCity(customer.GetAddress());
        if (!_customersByCity.ContainsKey(city))
            _customersByCity[city] = new List<Customer>();
        _customersByCity[city].Add(customer);
        
        _customerEmails.Add(customer.GetEmail().ToLower());
        
        _registrationQueue.Enqueue(customer);
        
        Console.WriteLine($"Клиент {customer.GetName()} добавлен в репозиторий");
    }
    
    public Customer GetCustomerById(int id)
    {
        if (_customersById.TryGetValue(id, out Customer customer))
        {
            _lastViewed.Push(customer);
            return customer;
        }
        return null;
    }
    
    public List<Customer> GetCustomersByCity(string city)
    {
        return _customersByCity.ContainsKey(city) ? _customersByCity[city] : new List<Customer>();
    }
    
    public bool IsEmailUnique(string email)
    {
        return !_customerEmails.Contains(email.ToLower());
    }
    
    public Customer ProcessNextRegistration()
    {
        return _registrationQueue.Count > 0 ? _registrationQueue.Dequeue() : null;
    }
    
    public Customer GetLastViewedCustomer()
    {
        return _lastViewed.Count > 0 ? _lastViewed.Peek() : null;
    }
    
    public void MarkAsActive(Customer customer)
    {
        _recentlyActive.RemoveAll(c => c.GetCustomerId() == customer.GetCustomerId());
        _recentlyActive.Insert(0, customer);
        
        if (_recentlyActive.Count > 10)
            _recentlyActive = _recentlyActive.Take(10).ToList();
    }
    
    public IEnumerable<Customer> FindCustomers(Func<Customer, bool> predicate)
    {
        return _customersById.Values.Where(predicate);
    }
    
    public void DisplayStatistics()
    {
        Console.WriteLine("\n=== СТАТИСТИКА КЛИЕНТОВ ===");
        Console.WriteLine($"Всего клиентов: {_customersById.Count}");
        Console.WriteLine($"Уникальных email: {_customerEmails.Count}");
        Console.WriteLine($"Городов в базе: {_customersByCity.Count}");
        Console.WriteLine($"В очереди регистрации: {_registrationQueue.Count}");
        Console.WriteLine($"Недавно активных: {_recentlyActive.Count}");
        
        var topCities = _customersByCity
            .OrderByDescending(kv => kv.Value.Count)
            .Take(5);
            
        Console.WriteLine("\nТоп 5 городов:");
        foreach (var city in topCities)
        {
            Console.WriteLine($"- {city.Key}: {city.Value.Count} клиентов");
        }
    }
    
    private string ExtractCity(string address)
    {
        if (string.IsNullOrEmpty(address)) return "Не указан";
        var parts = address.Split(',');
        return parts.Length > 0 ? parts[0].Trim() : address;
    }
}

public class AdvancedCustomerManager
{
    private CustomerRepository _repository;
    private CustomerEventService _eventService;
    private List<Action<Customer>> _customerActions;
    
    public AdvancedCustomerManager()
    {
        _repository = new CustomerRepository();
        _eventService = new CustomerEventService();
        _customerActions = new List<Action<Customer>>();
        
        SetupEventHandlers();
    }
    
    private void SetupEventHandlers()
    {
        _eventService.CustomerRegistered += (sender, e) => 
        {
            Console.WriteLine($"[СОБЫТИЕ] {e.Message} в {e.EventTime:HH:mm:ss}");
        };
        
        _eventService.CustomerUpdated += (sender, e) => 
        {
            Console.WriteLine($"[СОБЫТИЕ] {e.Message} в {e.EventTime:HH:mm:ss}");
        };
        
        _eventService.PurchaseMade += (sender, e) => 
        {
            Console.WriteLine($"[ПОКУПКА] Совершена покупка {e.Product} на {e.Amount:N2} в {e.PurchaseTime:HH:mm:ss}");
        };
        
        _eventService.BulkNotificationSent += (message, customer) => 
        {
            Console.WriteLine($"[РАССЫЛКА] {customer.GetName()}: {message}");
        };
    }
    
    public void RegisterCustomer(Customer customer)
    {
        _repository.AddCustomer(customer);
        _eventService.OnCustomerRegistered(customer);
        
        foreach (var action in _customerActions)
        {
            action(customer);
        }
    }
    
    public void UpdateCustomer(Customer customer)
    {
        _repository.MarkAsActive(customer);
        _eventService.OnCustomerUpdated(customer);
    }
    
    public void RecordPurchase(Customer customer, double amount, string product)
    {
        _eventService.OnPurchaseMade(customer, amount, product);
        customer.MakePurchase(amount);
    }
    
    public void AddCustomerAction(Action<Customer> action)
    {
        _customerActions.Add(action);
    }
    
    public void DisplayCustomerAnalytics()
    {
        var customers = _repository.FindCustomers(c => true).ToList();
        
        if (customers.Any())
        {
            Console.WriteLine("\n=== АНАЛИТИКА КЛИЕНТОВ ===");
            
            var averageAge = customers.Average(c => c.CalculateAge());
            var maxAge = customers.Max(c => c.CalculateAge());
            var minAge = customers.Min(c => c.CalculateAge());
            
            Console.WriteLine($"Возраст: средний {averageAge:F1}, от {minAge} до {maxAge} лет");
            
            var vipCount = customers.Count(c => c is VipCustomer);
            var regularCount = customers.Count(c => c is RegularCustomer);
            var groupCount = customers.Count(c => c is GroupCustomer);
            
            Console.WriteLine($"Типы клиентов: VIP - {vipCount}, Обычные - {regularCount}, Групповые - {groupCount}");
            
            var ageGroups = customers
                .GroupBy(c => c.CalculateAge() / 10 * 10)
                .OrderBy(g => g.Key);
                
            Console.WriteLine("\nРаспределение по возрасту:");
            foreach (var group in ageGroups)
            {
                Console.WriteLine($"- {group.Key}-{group.Key + 9} лет: {group.Count()} клиентов");
            }
        }
    }
    
    public CustomerRepository Repository => _repository;
}

public class Customer : INotifiable, IAuthenticable, IPurchasable
{
    private int _customerId;
    private string _name;
    private string _email;
    private DateTime _birthDate;
    private string _address;
    private string _gender;
    private List<string> _preferences;
    private string _nationality;
    private string _occupation;
    private decimal _creditLimit;
    private bool _isActive;

    private string _phoneNumber;
    private bool _isSubscribed;
    private string _password;
    private DateTime _lastLogin;
    private List<double> _purchases;

    string INotifiable.PhoneNumber 
    { 
        get => _phoneNumber;
        set => _phoneNumber = value;
    }

    bool INotifiable.IsSubscribed 
    { 
        get => _isSubscribed;
        set => _isSubscribed = value;
    }

    void INotifiable.SendNotification(string message)
    {
        if (_isSubscribed)
        {
            Console.WriteLine($"Уведомление для {_name}: {message}");
        }
    }

    DateTime IAuthenticable.LastLogin 
    { 
        get => _lastLogin;
        set => _lastLogin = value;
    }

    bool IAuthenticable.Authenticate(string password)
    {
        bool isAuthenticated = _password == password;
        if (isAuthenticated)
        {
            _lastLogin = DateTime.Now;
            Console.WriteLine($"Клиент {_name} успешно аутентифицирован");
        }
        return isAuthenticated;
    }

    void IAuthenticable.ChangePassword(string newPassword)
    {
        if (newPassword.Length >= 6)
        {
            _password = newPassword;
            Console.WriteLine("Пароль успешно изменен");
        }
    }

    void IPurchasable.MakePurchase(double amount)
    {
        _purchases.Add(amount);
        Console.WriteLine($"Покупка на сумму {amount:N2} завершена");
    }

    double IPurchasable.GetTotalSpent()
    {
        double total = 0;
        foreach (var purchase in _purchases)
        {
            total += purchase;
        }
        return total;
    }

    int IPurchasable.GetPurchaseCount()
    {
        return _purchases.Count;
    }

    public Customer(int customerId, string name, string email, DateTime birthDate, string address, string gender)
    {
        _customerId = customerId;
        _name = name;
        _email = email;
        _birthDate = birthDate;
        _address = address;
        _gender = gender;
        _preferences = new List<string>();
        _nationality = "Россия";
        _occupation = "Не указано";
        _creditLimit = 10000;
        _isActive = true;
        _phoneNumber = "";
        _isSubscribed = true;
        _password = "default123";
        _lastLogin = DateTime.MinValue;
        _purchases = new List<double>();
    }

    public string GetNationality() => _nationality;
    public void SetNationality(string value) => _nationality = value;

    public string GetOccupation() => _occupation;
    public void SetOccupation(string value) => _occupation = value;

    public decimal GetCreditLimit() => _creditLimit;
    public virtual void SetCreditLimit(decimal value)
    {
        if (value >= 0)
        {
            _creditLimit = value;
            Console.WriteLine($"Кредитный лимит установлен: {value:C}");
        }
    }

    public bool GetIsActive() => _isActive;
    public void SetIsActive(bool value)
    {
        _isActive = value;
        Console.WriteLine($"Статус активности: {(value ? "активен" : "неактивен")}");
    }

    public int GetCustomerId() => _customerId;
    public void SetCustomerId(int value) => _customerId = value;

    public string GetName() => _name;
    public void SetName(string value) => _name = value;

    public string GetEmail() => _email;
    public virtual void SetEmail(string value)
    {
        if (!string.IsNullOrEmpty(value))
        {
            _email = value;
            Console.WriteLine($"Email обновлен: {_email}");
        }
    }

    public DateTime GetBirthDate() => _birthDate;
    public void SetBirthDate(DateTime value) => _birthDate = value;

    public string GetAddress() => _address;
    public void SetAddress(string value) => _address = value;

    public string GetGender() => _gender;
    public void SetGender(string value) => _gender = value;

    public List<string> GetPreferences() => _preferences;

    public virtual void AddPreference(string preference)
    {
        if (!_preferences.Contains(preference))
        {
            _preferences.Add(preference);
            Console.WriteLine($"Добавлено предпочтение: {preference}");
        }
    }

    public void RemovePreference(string preference)
    {
        if (_preferences.Contains(preference))
        {
            _preferences.Remove(preference);
            Console.WriteLine($"Удалено предпочтение: {preference}");
        }
    }

    public int CalculateAge()
    {
        var today = DateTime.Today;
        var age = today.Year - _birthDate.Year;
        if (_birthDate.Date > today.AddYears(-age)) age--;
        return age;
    }

    public virtual void ApplyDiscount(double amount)
    {
        Console.WriteLine($"Базовая скидка {amount}% применена для {_name}");
    }

    public virtual void GenerateReport()
    {
        Console.WriteLine($"Отчет по клиенту {_name}:");
        Console.WriteLine($"- ID: {_customerId}");
        Console.WriteLine($"- Email: {_email}");
        Console.WriteLine($"- Национальность: {_nationality}");
        Console.WriteLine($"- Профессия: {_occupation}");
    }

    public void UpdateOccupation(string newOccupation)
    {
        _occupation = newOccupation;
        Console.WriteLine($"Профессия обновлена: {newOccupation}");
    }

    public virtual void DisplayProfile()
    {
        Console.WriteLine($"ID клиента: {_customerId}");
        Console.WriteLine($"Имя клиента: {_name}");
        Console.WriteLine($"Email: {_email}");
        Console.WriteLine($"Дата рождения: {_birthDate:dd.MM.yyyy}");
        Console.WriteLine($"Возраст: {CalculateAge()} лет");
        Console.WriteLine($"Адрес: {_address}");
        Console.WriteLine($"Пол: {_gender}");
        Console.WriteLine($"Национальность: {_nationality}");
        Console.WriteLine($"Профессия: {_occupation}");
        Console.WriteLine($"Кредитный лимит: {_creditLimit:C}");
        Console.WriteLine($"Статус: {(_isActive ? "Активен" : "Неактивен")}");
        
        if (_preferences.Count > 0)
        {
            Console.WriteLine($"Предпочтения: {string.Join(", ", _preferences)}");
        }
    }

    public virtual void GetFullName()
    {
        Console.WriteLine($"Полное имя: {_name}");
    }

    public virtual void UpdateEmail(string newEmail)
    {
        SetEmail(newEmail);
    }

    public void InteractWith(Customer other)
    {
        Console.WriteLine($"{_name} взаимодействует с {other.GetName()}");
    }

    public virtual void DisplayCustomerCategory()
    {
        int age = CalculateAge();
        if (age < 25)
            Console.WriteLine("Категория: Молодой клиент");
        else if (age < 60)
            Console.WriteLine("Категория: Взрослый клиент");
        else
            Console.WriteLine("Категория: Клиент старшего возраста");
    }

    public void SendNotification(string message)
    {
        ((INotifiable)this).SendNotification(message);
    }

    public bool Authenticate(string password)
    {
        return ((IAuthenticable)this).Authenticate(password);
    }

    public void ChangePassword(string newPassword)
    {
        ((IAuthenticable)this).ChangePassword(newPassword);
    }

    public void MakePurchase(double amount)
    {
        ((IPurchasable)this).MakePurchase(amount);
    }

    public double GetTotalSpent()
    {
        return ((IPurchasable)this).GetTotalSpent();
    }

    public int GetPurchaseCount()
    {
        return ((IPurchasable)this).GetPurchaseCount();
    }

    public void RecordActivity()
    {
        Console.WriteLine($"Активность клиента {_name} записана");
    }
    
    public virtual string GetCustomerType()
    {
        return "Базовый клиент";
    }
}

public class VipCustomer : Customer
{
    private int _loyaltyPoints;
    private string _vipLevel;
    private string _personalManager;
    private DateTime _vipSince;
    private List<string> _exclusiveOffers;

    public VipCustomer(int customerId, string name, string email, DateTime birthDate, 
                      string address, string gender, int loyaltyPoints, string vipLevel)
        : base(customerId, name, email, birthDate, address, gender)
    {
        _loyaltyPoints = loyaltyPoints;
        _vipLevel = vipLevel;
        _personalManager = "Не назначен";
        _vipSince = DateTime.Now;
        _exclusiveOffers = new List<string>();
    }

    public string GetPersonalManager() => _personalManager;
    public void SetPersonalManager(string value)
    {
        _personalManager = value;
        Console.WriteLine($"Персональный менеджер установлен: {value}");
    }

    public DateTime GetVipSince() => _vipSince;
    public List<string> GetExclusiveOffers() => _exclusiveOffers;

    public int GetLoyaltyPoints() => _loyaltyPoints;
    public void SetLoyaltyPoints(int value) => _loyaltyPoints = value;

    public string GetVipLevel() => _vipLevel;
    public void SetVipLevel(string value) => _vipLevel = value;

    public void AddExclusiveOffer(string offer)
    {
        _exclusiveOffers.Add(offer);
        Console.WriteLine($"Добавлено эксклюзивное предложение: {offer}");
    }

    public void DisplayExclusiveOffers()
    {
        Console.WriteLine($"Эксклюзивные предложения для {GetName()}:");
        foreach (var offer in _exclusiveOffers)
        {
            Console.WriteLine($" - {offer}");
        }
    }

    public override void SetCreditLimit(decimal value)
    {
        decimal maxLimit = _vipLevel == "Platinum" ? 100000 : 50000;
        if (value <= maxLimit)
        {
            base.SetCreditLimit(value);
        }
        else
        {
            Console.WriteLine($"Превышен максимальный лимит для уровня {_vipLevel}: {maxLimit:C}");
        }
    }

    public override void DisplayProfile()
    {
        base.DisplayProfile();
        Console.WriteLine($"Баллы лояльности: {_loyaltyPoints}");
        Console.WriteLine($"Уровень VIP: {_vipLevel}");
        Console.WriteLine($"Персональный менеджер: {_personalManager}");
        Console.WriteLine($"VIP с: {_vipSince:dd.MM.yyyy}");
        Console.WriteLine("Статус: VIP клиент");
        
        if (_exclusiveOffers.Count > 0)
        {
            DisplayExclusiveOffers();
        }
    }

    public override void AddPreference(string preference)
    {
        base.AddPreference(preference);
        _loyaltyPoints += 10; 
        Console.WriteLine($"Начислено 10 бонусных баллов за предпочтение");
    }

    public override void ApplyDiscount(double amount)
    {
        double vipDiscount = amount * 1.5; 
        Console.WriteLine($"VIP скидка {vipDiscount}% применена для {GetName()}");
    }

    public override void GenerateReport()
    {
        base.GenerateReport();
        Console.WriteLine($"- Уровень VIP: {_vipLevel}");
        Console.WriteLine($"- Баллы лояльности: {_loyaltyPoints}");
        Console.WriteLine($"- Персональный менеджер: {_personalManager}");
    }

    public void TransferPoints(VipCustomer target, int points)
    {
        if (points > 0 && _loyaltyPoints >= points)
        {
            _loyaltyPoints -= points;
            target.SetLoyaltyPoints(target.GetLoyaltyPoints() + points);
            Console.WriteLine($"Передано {points} баллов клиенту {target.GetName()}");
        }
    }

    public override void DisplayCustomerCategory()
    {
        base.DisplayCustomerCategory();
        Console.WriteLine($"VIP статус: {_vipLevel} (с {_vipSince:yyyy})");
    }

    public override string GetCustomerType()
    {
        return $"VIP клиент ({_vipLevel})";
    }
}

public class RegularCustomer : Customer
{
    private DateTime _registrationDate;
    private DateTime _lastEmailUpdate;
    private int _loginAttempts;
    private string _securityQuestion;
    private bool _emailVerified;

    public RegularCustomer(int customerId, string name, string email, DateTime birthDate,
                          string address, string gender, DateTime registrationDate)
        : base(customerId, name, email, birthDate, address, gender)
    {
        _registrationDate = registrationDate;
        _lastEmailUpdate = DateTime.MinValue;
        _loginAttempts = 0;
        _securityQuestion = "Любимое блюдо";
        _emailVerified = false;
    }

    public string GetSecurityQuestion() => _securityQuestion;
    public void SetSecurityQuestion(string value) => _securityQuestion = value;

    public bool GetEmailVerified() => _emailVerified;
    public void VerifyEmail()
    {
        _emailVerified = true;
        Console.WriteLine("Email успешно подтвержден");
    }

    public DateTime GetRegistrationDate() => _registrationDate;
    public void SetRegistrationDate(DateTime value) => _registrationDate = value;

    public DateTime GetLastEmailUpdate() => _lastEmailUpdate;
    public void SetLastEmailUpdate(DateTime value) => _lastEmailUpdate = value;

    public void ResetLoginAttempts()
    {
        _loginAttempts = 0;
        Console.WriteLine("Счетчик попыток входа сброшен");
    }

    public void RequestPasswordReset()
    {
        Console.WriteLine("Запрос на сброс пароля отправлен на email");
    }

    public override void SetEmail(string value)
    {
        base.SetEmail(value);
        _lastEmailUpdate = DateTime.Now;
        _emailVerified = false; 
        Console.WriteLine($"Дата обновления email: {_lastEmailUpdate:dd.MM.yyyy HH:mm}");
        Console.WriteLine("Требуется подтверждение нового email");
    }

    public override void DisplayProfile()
    {
        base.DisplayProfile();
        Console.WriteLine($"Дата регистрации: {_registrationDate:dd.MM.yyyy}");
        if (_lastEmailUpdate != DateTime.MinValue)
        {
            Console.WriteLine($"Последнее обновление email: {_lastEmailUpdate:dd.MM.yyyy HH:mm}");
        }
        Console.WriteLine($"Подтверждение email: {(_emailVerified ? "Да" : "Нет")}");
        Console.WriteLine($"Контрольный вопрос: {_securityQuestion}");
        Console.WriteLine("Статус: Обычный клиент");
    }

    public void UpdateEmail(string newEmail, bool skipVerification)
    {
        SetEmail(newEmail);
        if (skipVerification)
        {
            _emailVerified = true;
            Console.WriteLine("Верификация email пропущена");
        }
    }

    public override void ApplyDiscount(double amount)
    {
        double regularDiscount = amount * 0.8; 
        Console.WriteLine($"Стандартная скидка {regularDiscount}% применена для {GetName()}");
    }

    public override void GenerateReport()
    {
        base.GenerateReport();
        Console.WriteLine($"- Дата регистрации: {_registrationDate:dd.MM.yyyy}");
        Console.WriteLine($"- Подтверждение email: {(_emailVerified ? "Да" : "Нет")}");
        Console.WriteLine($"- Контрольный вопрос: {_securityQuestion}");
    }

    public void SendInvitation(Customer target)
    {
        Console.WriteLine($"{GetName()} отправляет приглашение {target.GetName()}");
    }

    public int GetDaysSinceRegistration()
    {
        return (DateTime.Now - _registrationDate).Days;
    }

    public override void DisplayCustomerCategory()
    {
        base.DisplayCustomerCategory();
        int daysRegistered = GetDaysSinceRegistration();
        if (daysRegistered > 365)
            Console.WriteLine("Статус: Постоянный клиент");
        else
            Console.WriteLine("Статус: Новый клиент");
    }

    public override string GetCustomerType()
    {
        return $"Обычный клиент (зарегистрирован {GetDaysSinceRegistration()} дней назад)";
    }
}

public class GroupCustomer : Customer
{
    private string _groupName;
    private List<Customer> _groupMembers;
    private string _companyType;
    private string _taxId;
    private decimal _groupDiscount;

    public GroupCustomer(int customerId, string name, string email, DateTime birthDate,
                        string address, string gender, string groupName)
        : base(customerId, name, email, birthDate, address, gender)
    {
        _groupName = groupName;
        _groupMembers = new List<Customer>();
        _companyType = "ООО";
        _taxId = "Не указан";
        _groupDiscount = 5.0m;
    }

    public string GetCompanyType() => _companyType;
    public void SetCompanyType(string value) => _companyType = value;

    public string GetTaxId() => _taxId;
    public void SetTaxId(string value)
    {
        _taxId = value;
        Console.WriteLine($"ИНН установлен: {value}");
    }

    public decimal GetGroupDiscount() => _groupDiscount;
    public void SetGroupDiscount(decimal value)
    {
        _groupDiscount = value;
        Console.WriteLine($"Групповая скидка установлена: {value}%");
    }

    public string GetGroupName() => _groupName;
    public void SetGroupName(string value) => _groupName = value;

    public List<Customer> GetGroupMembers() => _groupMembers;

    public void CalculateGroupDiscount()
    {
        decimal discount = _groupMembers.Count * 0.5m + 5.0m;
        SetGroupDiscount(Math.Min(discount, 20.0m)); 
    }

    public void GenerateGroupInvoice(double amount)
    {
        double discountedAmount = amount * (1 - (double)_groupDiscount / 100);
        Console.WriteLine($"Счет для группы {_groupName}:");
        Console.WriteLine($"- Сумма: {amount:N2}");
        Console.WriteLine($"- Скидка: {_groupDiscount}%");
        Console.WriteLine($"- Итого: {discountedAmount:N2}");
    }

    public override void GetFullName()
    {
        Console.WriteLine($"Название группы: {_groupName}");
        Console.WriteLine($"Юридическое лицо: {_companyType}");
        Console.WriteLine($"ИНН: {_taxId}");
    }

    public override void DisplayProfile()
    {
        Console.WriteLine("Групповой клиент");
        Console.WriteLine($"Название группы: {_groupName}");
        Console.WriteLine($"Тип компании: {_companyType}");
        Console.WriteLine($"ИНН: {_taxId}");
        Console.WriteLine($"Групповая скидка: {_groupDiscount}%");
        base.DisplayProfile();
        Console.WriteLine($"Количество участников: {_groupMembers.Count + 1}"); 
    }

    public void MakePurchase(double amount, string purchaseType)
    {
        double discountedAmount = amount * (1 - (double)_groupDiscount / 100);
        Console.WriteLine($"Групповая покупка ({purchaseType}) на сумму {amount:N2} завершена");
        Console.WriteLine($"Применена скидка {_groupDiscount}%, итого: {discountedAmount:N2}");
    }

    public override void ApplyDiscount(double amount)
    {
        double groupDiscount = amount + (double)_groupDiscount;
        Console.WriteLine($"Групповая скидка {groupDiscount}% применена для {_groupName}");
    }

    public override void GenerateReport()
    {
        Console.WriteLine($"Отчет по групповому клиенту {_groupName}:");
        Console.WriteLine($"- ID: {GetCustomerId()}");
        Console.WriteLine($"- Тип компании: {_companyType}");
        Console.WriteLine($"- ИНН: {_taxId}");
        Console.WriteLine($"- Групповая скидка: {_groupDiscount}%");
        Console.WriteLine($"- Количество участников: {_groupMembers.Count + 1}");
    }

    public void AddMember(Customer newMember)
    {
        if (!_groupMembers.Contains(newMember))
        {
            _groupMembers.Add(newMember);
            Console.WriteLine($"Добавление {newMember.GetName()} в группу {_groupName}");
            CalculateGroupDiscount(); 
        }
    }

    public void RemoveMember(Customer member)
    {
        if (_groupMembers.Contains(member))
        {
            _groupMembers.Remove(member);
            Console.WriteLine($"Удаление {member.GetName()} из группы {_groupName}");
            CalculateGroupDiscount();
        }
    }

    public override void DisplayCustomerCategory()
    {
        Console.WriteLine($"Категория: Групповой клиент ({_groupMembers.Count + 1} участников)");
        Console.WriteLine($"Тип: {_companyType}, Скидка: {_groupDiscount}%");
    }

    public override string GetCustomerType()
    {
        return $"Групповой клиент ({_groupMembers.Count + 1} участников)";
    }
}
        var notificationService = new EmailNotificationService();
        var authService = new BasicAuthenticationService();
        var purchaseService = new PurchaseProcessingService();

        var customerManager = new CustomerManager<Customer>(notificationService, authService);
        var vipManager = new CustomerManager<VipCustomer>(notificationService, authService);
        var loyaltySystem = new LoyaltySystem<Customer>(purchaseService);
        
        var advancedManager = new AdvancedCustomerManager();
        
        Customer customer = new Customer(1, "Иван Иванов", "ivan@example.com", 
                                       new DateTime(1985, 5, 15), "Москва, ул. Ленина, 123", "Мужской");
        
        VipCustomer vipCustomer = new VipCustomer(2, "Петр Петров", "petr@example.com", 
                                                new DateTime(1978, 8, 22), "Санкт-Петербург, ул. Центральная, 45", 
                                                "Мужской", 150, "Platinum");
        
        RegularCustomer regularCustomer = new RegularCustomer(3, "Мария Сидорова", "maria@example.com", 
                                                            new DateTime(1990, 3, 10), "Москва, пр. Мира, 67", 
                                                            "Женский", new DateTime(2025, 5, 15));
        
        GroupCustomer groupCustomer = new GroupCustomer(4, "Алексей Иванов", "alex@company.com", 
                                                      new DateTime(1980, 12, 5), "Екатеринбург, ул. Промышленная, 89", 
                                                      "Мужской", "ООО ТехноПлюс");

        Console.WriteLine(" РЕГИСТРАЦИЯ В РАСШИРЕННОЙ СИСТЕМЕ ");
        advancedManager.RegisterCustomer(customer);
        advancedManager.RegisterCustomer(vipCustomer);
        advancedManager.RegisterCustomer(regularCustomer);
        advancedManager.RegisterCustomer(groupCustomer);
        
        advancedManager.AddCustomerAction(c => 
        {
            Console.WriteLine($"Выполняется действие для нового клиента: {c.GetName()}");
            c.AddPreference("Автоматически добавленное предпочтение");
        });
        
        advancedManager.AddCustomerAction(c => 
        {
            if (c is VipCustomer vip)
            {
                vip.AddExclusiveOffer("Добро пожаловать в VIP программу!");
            }
        });
        
        Customer newCustomer = new Customer(5, "Новый Клиент", "new@example.com", 
                                          new DateTime(1995, 1, 1), "Казань, ул. Новая, 1", "Мужской");
        advancedManager.RegisterCustomer(newCustomer);
        
        Console.WriteLine("\n РАБОТА С КОЛЛЕКЦИЯМИ ");
        advancedManager.Repository.DisplayStatistics();
        
        var moscowCustomers = advancedManager.Repository.GetCustomersByCity("Москва");
        Console.WriteLine($"\nКлиенты в Москве: {moscowCustomers.Count}");
        
        var youngCustomers = advancedManager.Repository.FindCustomers(c => c.CalculateAge() < 30);
        Console.WriteLine($"Молодые клиенты (до 30 лет): {youngCustomers.Count()}");
        
        var highLimitCustomers = advancedManager.Repository.FindCustomers(c => c.GetCreditLimit() > 20000);
        Console.WriteLine($"Клиенты с высоким лимитом: {highLimitCustomers.Count()}");
        
        Console.WriteLine("\n ДЕМОНСТРАЦИЯ СОБЫТИЙ ");
        advancedManager.UpdateCustomer(vipCustomer);
        advancedManager.RecordPurchase(vipCustomer, 2500.75, "Смартфон");
        advancedManager.RecordPurchase(regularCustomer, 500.00, "Книги");
        
        Console.WriteLine("\n ОБРАБОТКА ОЧЕРЕДИ РЕГИСТРАЦИИ ");
        var nextCustomer = advancedManager.Repository.ProcessNextRegistration();
        if (nextCustomer != null)
        {
            Console.WriteLine($"Обрабатывается клиент из очереди: {nextCustomer.GetName()}");
        }
        
        advancedManager.DisplayCustomerAnalytics();
        
        customerManager.AddCustomer(customer);
        vipManager.AddCustomer(vipCustomer);
        customerManager.AddCustomer(regularCustomer);
        customerManager.AddCustomer(groupCustomer);

        customer.SetOccupation("Инженер");
        customer.SetNationality("Россия");
        customer.SetCreditLimit(15000);

        vipCustomer.SetPersonalManager("Анна Петрова");
        vipCustomer.SetCreditLimit(75000);
        vipCustomer.AddExclusiveOffer("Персональная консультация");
        vipCustomer.AddExclusiveOffer("Ранний доступ к новинкам");

        regularCustomer.SetSecurityQuestion("Имя домашнего питомца");
        regularCustomer.VerifyEmail();

        groupCustomer.SetCompanyType("ООО");
        groupCustomer.SetTaxId("1234567890");
        groupCustomer.SetGroupDiscount(10.0m);

        customer.AddPreference("Электроника");
        vipCustomer.AddPreference("Премиум товары");
        regularCustomer.AddPreference("Распродажи");

        Console.WriteLine("\n ДЕМОНСТРАЦИЯ РАБОТЫ СИСТЕМЫ ");

        Customer[] customers = { customer, vipCustomer, regularCustomer, groupCustomer };
        
        foreach (var cust in customers)
        {
            Console.WriteLine("\n" + new string('-', 30));
            cust.GetFullName();
            cust.DisplayProfile(); 
            cust.ApplyDiscount(10);
            cust.GenerateReport(); 
            cust.DisplayCustomerCategory();
        }

        Console.WriteLine("\n СИСТЕМА УВЕДОМЛЕНИЙ ");
        customerManager.SendBulkNotification("Специальное предложение для всех клиентов!");

        Console.WriteLine("\n СИСТЕМА АУТЕНТИФИКАЦИИ ");
        customerManager.AuthenticateCustomer(customer, "default123");
        customerManager.AuthenticateCustomer(vipCustomer, "wrongpassword");

        Console.WriteLine("\n СИСТЕМА ЛОЯЛЬНОСТИ И ПОКУПОК ");
        loyaltySystem.ProcessPurchase(vipCustomer, 1500.50);
        loyaltySystem.ProcessPurchase(regularCustomer, 500.00);
        loyaltySystem.ProcessPurchase(groupCustomer, 5000.00);

        Console.WriteLine($"\nБаллы VIP клиента: {loyaltySystem.GetPoints(vipCustomer)}");
        Console.WriteLine($"Баллы обычного клиента: {loyaltySystem.GetPoints(regularCustomer)}");

        Console.WriteLine("\n ДОПОЛНИТЕЛЬНЫЕ ВОЗМОЖНОСТИ ");

        VipCustomer recipient = new VipCustomer(5, "Новый VIP клиент", "newvip@example.com", 
                                              new DateTime(1982, 7, 30), "ул. Новая, 1", 
                                              "Мужской", 50, "Gold");
        
        vipCustomer.TransferPoints(recipient, 30);
        regularCustomer.SendInvitation(customer);
        groupCustomer.AddMember(regularCustomer);
        groupCustomer.AddMember(customer);

        customer.SendNotification("Тестовое уведомление");
        customer.Authenticate("default123");
        customer.ChangePassword("newPassword123");
        customer.MakePurchase(100.50);

        Console.WriteLine($"Общие расходы клиента: {customer.GetTotalSpent():N2}");
        Console.WriteLine($"Количество покупок: {customer.GetPurchaseCount()}");

        Console.WriteLine("\n ВСЕ КЛИЕНТЫ В СИСТЕМЕ");
        customerManager.DisplayAllCustomers();

 РЕГИСТРАЦИЯ В РАСШИРЕННОЙ СИСТЕМЕ 
Клиент Иван Иванов добавлен в репозиторий
[СОБЫТИЕ] Зарегистрирован новый клиент: Иван Иванов в 22:56:47
Клиент Петр Петров добавлен в репозиторий
[СОБЫТИЕ] Зарегистрирован новый клиент: Петр Петров в 22:56:47
Клиент Мария Сидорова добавлен в репозиторий
[СОБЫТИЕ] Зарегистрирован новый клиент: Мария Сидорова в 22:56:47
Клиент Алексей Иванов добавлен в репозиторий
[СОБЫТИЕ] Зарегистрирован новый клиент: Алексей Иванов в 22:56:47
Клиент Новый Клиент добавлен в репозиторий
[СОБЫТИЕ] Зарегистрирован новый клиент: Новый Клиент в 22:56:47
Выполняется действие для нового клиента: Новый Клиент
Добавлено предпочтение: Автоматически добавленное предпочтение

 РАБОТА С КОЛЛЕКЦИЯМИ 

=== СТАТИСТИКА КЛИЕНТОВ ===
Всего клиентов: 5
Уникальных email: 5
Городов в базе: 4
В очереди регистрации: 5
Недавно активных: 0

Топ 5 городов:
- Москва: 2 клиентов
- Санкт-Петербург: 1 клиентов
- Екатеринбург: 1 клиентов
- Казань: 1 клиентов

Клиенты в Москве: 2
Молодые клиенты (д